# Sentence Embeddings & Topic Modelling

Pipeline: Flan-T5 summarisation | MiniLM embeddings | FAISS vector store | BERTopic

In [ ]:
import os, warnings
from pathlib import Path

import polars as pl
import numpy as np
import torch
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

CWD = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
NEWS_PATH = CWD / 'data' / 'Stock_news' / 'subset_news.parquet'
MODEL_DIR = CWD / 'models'
MODEL_DIR.mkdir(exist_ok=True)

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
POOL = [
    'AAPL','MSFT','GOOG','GOOGL','AMZN','TSLA','META','NVDA','AMD','INTC','CRM','NFLX',
    'ADBE','PYPL','UBER','SQ','SHOP','ZM','SNAP','COIN','PLTR','ORCL',
    'QQQ','SPY','DIA','IWM',
    'T','VZ',
    'JPM','GS','MS','WFC','BAC','C',
    'XOM','CVX',
    'JNJ','PFE','MRNA','GILD','MRK','UNH','ABT',
    'WMT','COST','TGT','HD','KO','PEP','SBUX','MCD',
    'BA','GE','CAT','MMM',
    'DIS','CMCSA',
    'V','MA',
    'MU','QCOM','TXN','AVGO',
    'F','GM',
]
POOL_SET = set(POOL)

df = pl.read_parquet(NEWS_PATH)
df = df.filter(pl.col('Stock_symbol').is_in(POOL))
print(f'Full df: {df.shape[0]:,} rows')

In [ ]:
# ---- SAMPLE SIZE (change this when running on full data) ----
N_SAMPLE = 5
sample = df.sample(n=min(N_SAMPLE, df.shape[0]), seed=42)
print(f'Working sample: {sample.shape[0]} rows')
sample.head()

## 1. Summarisation with Flan-T5-small

Few-shot prompting with 2 examples. Compare against the existing `Lsa_summary` using ROUGE.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

FLAN_NAME = 'google/flan-t5-small'
FLAN_DIR = MODEL_DIR / 'flan-t5-small'

# Download once, then load from disk
if FLAN_DIR.exists():
    print(f'Loading Flan-T5 from {FLAN_DIR}')
    flan_tok = AutoTokenizer.from_pretrained(FLAN_DIR)
    flan_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_DIR)
else:
    print(f'Downloading {FLAN_NAME}...')
    flan_tok = AutoTokenizer.from_pretrained(FLAN_NAME)
    flan_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_NAME)
    flan_tok.save_pretrained(FLAN_DIR)
    flan_model.save_pretrained(FLAN_DIR)
    print(f'Saved to {FLAN_DIR}')

flan_model = flan_model.to(DEVICE).eval()
print('Flan-T5 ready')

In [ ]:
# Two few-shot examples baked into the prompt
FEW_SHOT_PROMPT = """Summarize the following news article in 2-3 sentences.

Example 1:
Article: Apple reported record quarterly revenue of $123.9 billion, up 11% year over year. \
iPhone revenue grew 9% to $71.4 billion. CEO Tim Cook said the company saw strong demand \
across all product categories and geographic segments.
Summary: Apple achieved record Q1 revenue of $123.9B driven by 9% iPhone growth. \
CEO Cook highlighted broad strength across products and regions.

Example 2:
Article: Tesla delivered 405,278 vehicles in Q4, missing analyst expectations of 420,760. \
The company cited logistical challenges and year-end demand shifts. Full-year deliveries \
reached 1.31 million, a 40% increase over the prior year.
Summary: Tesla Q4 deliveries of 405K missed estimates due to logistics issues. \
Full-year deliveries hit 1.31M, up 40% YoY.

Article: {article}
Summary:"""


def summarise_article(article, max_input=512, max_output=80):
    """Generate a summary using Flan-T5 with few-shot prompt."""
    prompt = FEW_SHOT_PROMPT.format(article=article[:1500])
    inputs = flan_tok(prompt, return_tensors='pt', max_length=max_input, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = flan_model.generate(**inputs, max_new_tokens=max_output, num_beams=2, early_stopping=True)
    return flan_tok.decode(out[0], skip_special_tokens=True)

In [ ]:
articles = sample['Article'].to_list()
lsa_summaries = sample['Lsa_summary'].to_list()

flan_summaries = []
for art in tqdm(articles, desc='Summarising'):
    flan_summaries.append(summarise_article(art))

# Show first result
print('Article (first 300 chars):', articles[0][:300])
print('\nFlan-T5 summary:', flan_summaries[0])
print('\nLSA summary (first 200 chars):', lsa_summaries[0][:200])

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Score Flan-T5 summaries against the full article (as reference)
flan_scores = [scorer.score(art, s) for art, s in zip(articles, flan_summaries)]
lsa_scores  = [scorer.score(art, s) for art, s in zip(articles, lsa_summaries)]

def avg_rouge(scores):
    """Average ROUGE F1 across samples."""
    return {
        k: np.mean([s[k].fmeasure for s in scores])
        for k in ['rouge1', 'rouge2', 'rougeL']
    }

print('Flan-T5 ROUGE:', {k: f'{v:.3f}' for k, v in avg_rouge(flan_scores).items()})
print('LSA    ROUGE:', {k: f'{v:.3f}' for k, v in avg_rouge(lsa_scores).items()})

## 2. Sentence Embeddings & FAISS

Embed chunked summaries with `all-MiniLM-L6-v2`, store in a FAISS index.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIR = MODEL_DIR / 'all-MiniLM-L6-v2'

if EMBED_DIR.exists():
    print(f'Loading embedding model from {EMBED_DIR}')
    embed_model = SentenceTransformer(str(EMBED_DIR), device=DEVICE)
else:
    print(f'Downloading {EMBED_NAME}...')
    embed_model = SentenceTransformer(EMBED_NAME, device=DEVICE)
    embed_model.save(str(EMBED_DIR))
    print(f'Saved to {EMBED_DIR}')

print('Embedding model ready')

In [ ]:
titles  = sample['Article_title'].to_list()
dates   = sample['Date'].to_list()
symbols = sample['Stock_symbol'].to_list()


def make_chunks(summary, title, date, symbol):
    """Split a summary into 2 chunks, each prefixed with metadata."""
    sentences = summary.split('. ')
    mid = max(1, len(sentences) // 2)
    chunk1 = '. '.join(sentences[:mid]) + '.'
    chunk2 = '. '.join(sentences[mid:]).strip()
    if not chunk2:
        chunk2 = chunk1  # fallback for single-sentence summaries

    prefix = f'[{symbol}] [{date}] {title} | '
    return [prefix + chunk1, prefix + chunk2]


# Build chunks from Flan-T5 summaries
chunks, chunk_meta = [], []
for i in range(len(flan_summaries)):
    for chunk in make_chunks(flan_summaries[i], titles[i], dates[i], symbols[i]):
        chunks.append(chunk)
        chunk_meta.append({'idx': i, 'symbol': symbols[i], 'date': dates[i], 'title': titles[i]})

print(f'{len(chunks)} chunks from {len(flan_summaries)} articles')
print('Sample chunk:', chunks[0][:200])

In [ ]:
import faiss

# Encode all chunks
embeddings = embed_model.encode(chunks, show_progress_bar=True, batch_size=32)
embeddings = embeddings.astype('float32')
print(f'Embeddings shape: {embeddings.shape}')

# Build FAISS index (flat L2 — exact search, fine for <1M vectors)
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f'FAISS index size: {index.ntotal}')

# Save index
FAISS_PATH = MODEL_DIR / 'news_embeddings.faiss'
faiss.write_index(index, str(FAISS_PATH))
print(f'Saved FAISS index to {FAISS_PATH}')

In [ ]:
# Quick retrieval test
query = 'tech company earnings report'
q_vec = embed_model.encode([query]).astype('float32')
D, I = index.search(q_vec, k=3)

print(f'Query: "{query}"\n')
for rank, (dist, idx) in enumerate(zip(D[0], I[0])):
    print(f'  #{rank+1} (dist={dist:.2f}): {chunks[idx][:150]}')

## 3. BERTopic — Topic Modelling

Cluster the embedded chunks into topics using UMAP + HDBSCAN.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

n_docs = len(chunks)

# Scale params to dataset size (small sample vs full run)
umap_neighbors = min(15, n_docs - 1)
umap_components = min(5, n_docs - 2)
hdbscan_min_cluster = max(2, min(50, n_docs // 5))

umap_model = UMAP(
    n_neighbors=umap_neighbors,
    n_components=umap_components,
    min_dist=0.0,
    metric='cosine',
    low_memory=True,
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=hdbscan_min_cluster,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embed_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=False,
    verbose=True
)

print(f'BERTopic config: {n_docs} docs, umap_neighbors={umap_neighbors}, '
      f'umap_components={umap_components}, hdbscan_min_cluster={hdbscan_min_cluster}')

# Fit on pre-computed embeddings
topics, probs = topic_model.fit_transform(chunks, embeddings=embeddings)
print(f'Found {len(set(topics)) - (1 if -1 in topics else 0)} topics (+ outlier cluster)')

In [ ]:
# Topic overview
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

In [ ]:
# Show top words for each topic (skip outlier -1)
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)[:5]]
    print(f'Topic {tid}: {" | ".join(words)}')

In [ ]:
# Save topic model
TOPIC_DIR = MODEL_DIR / 'bertopic_news'
topic_model.save(str(TOPIC_DIR), serialization='safetensors', save_ctfidf=True, save_embedding_model=EMBED_DIR)
print(f'Saved BERTopic model to {TOPIC_DIR}')